# Amazon ML Challenge 2026 — Google Colab Runtime Verification
## Business Entity Resolution

**Objective**: Verified, reproducible environment and hardware sanity check on Google Colab (GPU/CPU) before launching experimental workloads.

### Strict Rules:
1. **No Datasets in GitHub**: Datasets are mounted via Google Drive (`/content/drive/MyDrive/ML_AMAZON/student_resource/dataset`).
2. **Strict Verification**: Fails immediately if GPU is missing, Drive path is invalid, or any TSV is missing.
3. **Zero RAM Exhaustion**: Datasets are inspected via streaming chunked reads.
4. **No Full Pipeline / Model Training**: This notebook validates environment and micro-benchmarks only.

### Step 1: Verify Google Colab Runtime

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f"Running inside Google Colab: {IN_COLAB}")
if not IN_COLAB:
    print("[NOTICE] If running locally, please ensure paths and hardware settings match your local environment.")

### Step 2: GPU & PyTorch CUDA Availability Check
Ensures an active CUDA GPU is assigned (e.g., T4, V100, A100).

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "[ERROR] CUDA GPU is unavailable! Please switch runtime in Colab menu: "
    "Runtime -> Change runtime type -> Hardware accelerator: T4 GPU or A100 GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2)
cuda_ver = torch.version.cuda
torch_ver = torch.__version__

print(f"[OK] CUDA Available:     True")
print(f"[OK] GPU Device:         {gpu_name}")
print(f"[OK] Total VRAM:         {vram_gb} GB")
print(f"[OK] PyTorch Version:    {torch_ver}")
print(f"[OK] CUDA Version:       {cuda_ver}")

### Step 3: Clone / Sync GitHub Repository
Synchronizes source code from `https://github.com/Toxic012/ML_AMAZON-HACKTHON.git` and verifies `scripts/colab_runner.py`.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Toxic012/ML_AMAZON-HACKTHON.git"
REPO_NAME = "ML_AMAZON-HACKTHON"

if not os.path.exists(REPO_NAME) and not os.path.exists("code"):
    print(f"Cloning repository from {REPO_URL}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    %cd {REPO_NAME}

!git fetch origin
!git checkout main
!git pull origin main

current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print(f"[OK] Current Commit SHA: {current_commit}")

assert Path("scripts/colab_runner.py").exists(), "[ERROR] scripts/colab_runner.py missing from repository!"
print("[OK] Runner script verified: scripts/colab_runner.py")

### Step 4: Install Essential Dependencies

In [ ]:
!pip install -q polars lightgbm rapidfuzz sentence-transformers faiss-cpu psutil
print("[OK] Required packages installed.")

### Step 5: Mount Google Drive & Locate External Datasets
Locates the dataset directory at `/content/drive/MyDrive/ML_AMAZON/student_resource/dataset`.

In [ ]:
import os
from pathlib import Path

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted.")
except Exception as e:
    print(f"[INFO] Drive mount note: {e}")

EXPECTED_DRIVE_PATH = Path("/content/drive/MyDrive/ML_AMAZON/student_resource/dataset")
ALT_SCRATCH_PATH = Path("/content/dataset")
LOCAL_FALLBACK_PATH = Path("student_resource/dataset").resolve()

if EXPECTED_DRIVE_PATH.exists():
    DATA_DIR = EXPECTED_DRIVE_PATH
elif ALT_SCRATCH_PATH.exists():
    DATA_DIR = ALT_SCRATCH_PATH
elif LOCAL_FALLBACK_PATH.exists():
    DATA_DIR = LOCAL_FALLBACK_PATH
else:
    raise FileNotFoundError(
        f"[ERROR] DATASET_PATH_NOT_FOUND! Looked in:\n"
        f"  1. {EXPECTED_DRIVE_PATH}\n"
        f"  2. {ALT_SCRATCH_PATH}\n"
        f"  3. {LOCAL_FALLBACK_PATH}\n"
        f"Please ensure Google Drive is mounted and contains the student_resource/dataset directory."
    )

print(f"[OK] DATASET_PATH_FOUND: {DATA_DIR}")
os.environ["AMAZON_ML_DATA_DIR"] = str(DATA_DIR)

### Step 6: Stream-Audit All 7 TSV Dataset Files (Non-RAM Exhaustive)

In [ ]:
REQUIRED_FILES = [
    "train/train_source1.tsv",
    "train/train_source2.tsv",
    "train/train_source3.tsv",
    "train/train_ground_truth.tsv",
    "test/test_source1.tsv",
    "test/test_source2.tsv",
    "test/test_source3.tsv"
]

print(f"Verifying all 7 dataset files in {DATA_DIR}...")
total_size_mb = 0.0

for rel_path in REQUIRED_FILES:
    fpath = DATA_DIR / rel_path
    assert fpath.exists(), f"[ERROR] Required file missing: {fpath}"
    size_mb = round(fpath.stat().st_size / (1024 ** 2), 2)
    total_size_mb += size_mb
    
    # Read header and line count via stream chunking
    with open(fpath, "r", encoding="utf-8") as f:
        header = f.readline().strip().split("\t")
        
    lines = 0
    with open(fpath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            lines += chunk.count(b"\n")
            
    print(f"  [OK] {rel_path:30s} | {size_mb:8.2f} MB | {max(0, lines - 1):>9,} rows | cols: {len(header)}")

print(f"\n[OK] All 7 dataset files verified successfully! Total Size: {total_size_mb:.2f} MB")

### Step 7: Run Full Colab Environment Check & Benchmarking
Runs [scripts/colab_runner.py](file:///c:/Users/HP/Desktop/ML_AMAZON/scripts/colab_runner.py) to audit the full CPU 5-stage pipeline, GPU inference, and save machine-readable telemetry to `experiments/EXP-0001_colab_env_check/`.

In [ ]:
!python scripts/colab_runner.py --mode check --data-dir "{DATA_DIR}"

### Step 8: Standalone CUDA Tensor Microbenchmark
Executes a direct GPU matrix multiplication and synchronization test.

In [ ]:
import time
import torch

device = torch.device("cuda:0")
print(f"Running CUDA microbenchmark on {torch.cuda.get_device_name(0)}...")

# Warmup
_ = torch.matmul(torch.randn(500, 500, device=device), torch.randn(500, 500, device=device))
torch.cuda.synchronize()

# Benchmark 4000x4000 float32 matmul
t0 = time.perf_counter()
A = torch.randn(4000, 4000, device=device, dtype=torch.float32)
B = torch.randn(4000, 4000, device=device, dtype=torch.float32)
C = torch.matmul(A, B)
torch.cuda.synchronize()
t1 = time.perf_counter()

latency_ms = round((t1 - t0) * 1000, 2)
assert not torch.isnan(C).any(), "[ERROR] CUDA matmul produced NaN values!"
print(f"[OK] 4000x4000 Matrix Multiplication: {latency_ms} ms (Success: True)")

### Step 9: Display Telemetry Report (`experiments/EXP-0001_colab_env_check/`)

In [ ]:
import json
from pathlib import Path

report_path = Path("experiments/EXP-0001_colab_env_check/environment_check.json")
assert report_path.exists(), f"[ERROR] Report not found at {report_path}"

with open(report_path, "r", encoding="utf-8") as f:
    telemetry = json.load(f)

print("=" * 70)
print("VERIFIED COLAB ENVIRONMENT REPORT (EXP-0001):")
print("=" * 70)
print(json.dumps(telemetry, indent=2))
print("=" * 70)